## PARTE 1 — Sem Memória

### Carregando variáveis no código

In [11]:
from dotenv import load_dotenv
import os

load_dotenv()

# Credencial embedding e LLM
api_key = os.getenv("OCI_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# Informações do banco
ip_banco = os.getenv("IP_BANCO")
db_password = os.getenv("DB_PASSWORD")
db_name = "vetores"
db_user="appuser"
db_port=5432

### Função para chamar llm

In [12]:
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)

base_url = "https://inference.generativeai.us-chicago-1.oci.oraclecloud.com/20231130/actions/v1"  # ou sua URL
client_llm = OpenAI(base_url=base_url, api_key=api_key)

def perguntar(msg):
    response = client_llm.chat.completions.create(
        model="xai.grok-4-fast-non-reasoning",
        messages=[{"role": "user", "content": msg}]
    )
    return response.choices[0].message.content

### Teste 1 - sem memória

In [8]:
print(perguntar("Meu nome é Amanda."))
print(perguntar("Qual é meu nome?"))

Olá, Amanda! Prazer em conhecê-la. Como posso ajudar você hoje?
Não sei o seu nome, pois você não me disse. Como posso chamar você?


## PARTE 2 — Memória de Curto Prazo (Context Window)

### Salva iteração a cada pergunta

In [4]:
historico = []

def perguntar_com_memoria(msg):
    historico.append({"role": "user", "content": msg})

    response = client_llm.chat.completions.create(
        model="xai.grok-4-fast-non-reasoning",
        messages=historico
    )

    resposta = response.choices[0].message.content
    historico.append({"role": "assistant", "content": resposta})

    return resposta

### Teste 2 - Com memória no contexto 

In [73]:
print(perguntar_com_memoria("Meu nome é Amanda."))
print(perguntar_com_memoria("Qual é meu nome?"))

Olá de novo, Amanda! Se você quiser mudar isso ou bater um papo sobre algo específico, é só dizer. O que está na sua mente?
Seu nome é Amanda! O que mais você quer saber ou discutir? 😄


## PARTE 3 — Memória de Longo Prazo (Vector DB)

### Configurando banco

In [15]:
import psycopg
def get_conn():
    return psycopg.connect(
        host=ip_banco,
        dbname=db_name,
        user=db_user,
        password=db_password,
        port=db_port
    )

### Criando estrutura de memória longa

In [17]:
conn = get_conn()
cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS memoria_longa (
    id SERIAL PRIMARY KEY,
    texto TEXT,
    embedding VECTOR(1536)
);
""")
conn.commit()
conn.close()
cur.close()

### Artefato de salvar memória no banco

In [21]:
def salvar_memoria(texto):
    emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=texto
    ).data[0].embedding

    emb_str = "[" + ",".join(map(str, emb)) + "]"
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO memoria_longa (texto, embedding)
        VALUES (%s, %s::vector)
    """, (texto, emb_str))

    conn.commit()
    conn.close()
    cur.close()

### Buscar memória

In [22]:
def buscar_memoria(pergunta):
    emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=pergunta
    ).data[0].embedding

    emb_str = "[" + ",".join(map(str, emb)) + "]"
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("""
        SELECT texto
        FROM memoria_longa
        ORDER BY embedding <=> %s::vector
        LIMIT 3
    """, (emb_str,))
    

    result = [r[0] for r in cur.fetchall()] 
    conn.close()
    cur.close()
    return result

### Teste 3 - Busca de memória

In [23]:
salvar_memoria("Amanda trabalha na TechNova.")
salvar_memoria("Amanda é responsável por tickets P1.")

print(buscar_memoria("Onde Amanda trabalha?"))

['Amanda trabalha na TechNova.', 'Amanda é responsável por tickets P1.']


### Estrutura com resgate de memória longa

In [24]:
def perguntar_com_memoria_longa(pergunta):
    contexto = "\n".join(buscar_memoria(pergunta))

    prompt = f"""
    Contexto relevante:
    {contexto}

    Pergunta:
    {pergunta}
    """

    response = client_llm.chat.completions.create(
        model="xai.grok-4-fast-non-reasoning",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

### Teste 4 - Pergunta com resgate de contexto

In [26]:
print(perguntar_com_memoria_longa("Quem é Amanda?"))

Amanda é uma funcionária da TechNova, responsável por tickets de prioridade 1 (P1).


## PARTE 4 — Memória de Trabalho (Scratchpad)

### Resgatando funções do banco do arquivo 2.2 NL2SQL

In [30]:
def gerar_sql(pergunta):
    schema = """
    Tabela tickets:
    - id (integer)
    - cliente (text)
    - prioridade (text: P1, P2, P3, P4)
    - modulo (text)
    - status (text)
    - data_abertura (timestamp)
    - resolvido (boolean)
    """

    prompt = f"""
    Você é um especialista em SQL para PostgreSQL.
    Gere apenas a query SQL válida, sem explicação.

    {schema}

    Pergunta:
    {pergunta}
    
    Regras obrigatórias:
    - Sua resposta deve ser um SQL executável e somente isso
    - Remova qualquer informação adicional, marcadores, markdown e comentários da resposta (inclusive ```sql)
    - Avalie o script SQL antes de gerar a versão final
    """

    resposta = client_llm.chat.completions.create(
        model="xai.grok-4-fast-non-reasoning",
        messages=[{"role":"user","content":prompt}]
    )

    return resposta.choices[0].message.content.strip()

def executar_nl2sql(pergunta):
    sql = gerar_sql(pergunta)
    print("SQL gerado:", sql)
    conn = get_conn()
    cur = conn.cursor()

    cur.execute(sql)
    resultado = cur.fetchall()
    conn.close()
    cur.close()
    return resultado

### Estrutura de agente Scratchpad manual

In [31]:
def agente_com_scratchpad(pergunta):
    scratchpad = ""

    scratchpad += "Passo 1: Entender a pergunta.\n"
    scratchpad += f"Pergunta recebida: {pergunta}\n"

    if "P1" in pergunta:
        scratchpad += "Passo 2: Precisa consultar banco de tickets.\n"
        resultado = executar_nl2sql(pergunta)
        scratchpad += f"Resultado SQL: {resultado}\n"
    else:
        resultado = "Pergunta não relacionada a tickets."

    prompt = f"""
    Use as anotações internas abaixo para responder.

    Scratchpad:
    {scratchpad}

    Responda ao usuário:
    """

    response = client_llm.chat.completions.create(
        model="xai.grok-4-fast-non-reasoning",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

### Teste 5 - Agente Scratchpad

In [32]:
agente_com_scratchpad("Quantos tickets P1 existem?")

SQL gerado: SELECT COUNT(*) FROM tickets WHERE prioridade = 'P1';


'Existem 5 tickets P1.'

## Parte 4 - STM

### Instalando bibliotecas

In [35]:
!pip install openai tiktoken langchain-core langchain==1.1.0 langchain-openai==1.1.10 langchain-community


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\Amanda Machado\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Definindo estado

In [63]:
from pydantic import BaseModel
from typing import Optional

class UserUpdate(BaseModel):
    nome: Optional[str] = None
    tamanho_calcado: Optional[int] = None
    cor_favorita: Optional[str] = None
    nome_animal: Optional[str] = None

## Função de consultar estado

In [64]:
session_store = {}

def get_user_state(session_id):
    if session_id not in session_store:
        session_store[session_id] = UserUpdate()
    return session_store[session_id]

### Agente com STM

In [71]:
from langchain_openai import ChatOpenAI
import json

llm = ChatOpenAI(model="xai.grok-4-fast-non-reasoning", api_key=api_key, base_url=base_url)
structured_llm = llm.with_structured_output(UserUpdate)
def update_user_state(session_id, user_input):
    state = get_user_state(session_id)

    result = structured_llm.invoke(user_input)

    data = result.model_dump()

    for field, value in data.items():
        if value is not None:
            setattr(state, field, value)

    return state

### Fluxo de conversa

In [66]:
def conversar(session_id, user_input):
    
    state = update_user_state(session_id, user_input)
    
    prompt = f"""
    Você é um assistente vendedor.

    Estado atual do usuário:
    {state.model_dump()}

    Responda à mensagem:
    {user_input}
    """
    
    response = llm.invoke(prompt)
    return response.content

### Teste 6 - Simulando mensagens anteriores

In [67]:
session = "user_1"

print(conversar(session, "Oi, meu nome é Amanda"))
print(conversar(session, "Eu calço 36"))
print(conversar(session, "Minha cor favorita é verde"))
print(conversar(session, "Tenho um cachorro chamado Oliver"))

Olá, Amanda! Prazer em conhecê-la. Eu sou o assistente de vendas aqui da loja de calçados. Como posso ajudá-la hoje? Está procurando algo específico, como um par de sapatos confortável ou na sua cor favorita? 😊
Olá, Amanda! Ótimo saber que você calça 36. Isso nos ajuda a recomendar sapatos que fiquem perfeitos no seu pé. Você tem alguma cor favorita ou tipo de calçado em mente, como tênis, sandálias ou botas? Me conte mais para eu te ajudar melhor! 😊
Olá, Amanda! Que legal saber que sua cor favorita é verde – é uma cor tão vibrante e cheia de energia! Já atualizei isso no seu perfil. Você tem um pet? Me conta o nome dele, se quiser, para eu personalizar ainda mais as sugestões de produtos. O que você está procurando hoje? Algo em verde para combinar com seu estilo? 😊
Olá, Amanda! Que legal saber que você tem um cachorro chamado Oliver – adoro animais de estimação, eles trazem tanta alegria! Seu Oliver deve ser um companheiro incrível. Como eu sou um assistente vendedor, posso te ajudar

### Estado após as mensagens

In [68]:
get_user_state(session)

UserUpdate(nome='Amanda', tamanho_calcado=36, cor_favorita='verde', nome_animal='Oliver')

### Teste 7 - Exemplo de uso das informações de personalização

In [69]:
print(conversar(session, "Meu cachorro não está se sentindo muito bem, ainda da tempo de fazer o plano de saude pet de vocês?"))

Olá, Amanda! Sinto muito em saber que o Oliver não está se sentindo bem – espero que ele melhore logo. 😔

Sim, ainda dá tempo de ativar o plano de saúde pet! Na verdade, quanto mais cedo, melhor, para que o Oliver possa ter cobertura o mais rápido possível. Dependendo do plano que escolhermos, a ativação pode ser imediata ou em até 24 horas, cobrindo consultas, vacinas, exames e até emergências.

Me conta um pouquinho mais: que sintomas o Oliver está apresentando? Isso me ajuda a recomendar o plano ideal, como o nosso Pacote Essencial (a partir de R$ 49,90/mês) ou o Premium com cobertura mais ampla. E como você gosta de verde, posso sugerir opções com acessórios personalizados nessa cor para o pet!

O que acha de agendarmos uma consulta rápida para ativar? Estou aqui para ajudar! 🐶


### Teste 8 - Exemplo de uso das informações para complementação

In [72]:
print(conversar(session, "Quero aquele sapato Nike court todo branco, tem no estoque?"))

Olá, Amanda! Que bom que você está interessada no Nike Court Vision todo branco – é um modelo super estiloso e confortável, perfeito para o dia a dia. Pelo seu tamanho 36 e preferência por branco, ele combina demais com o seu estilo!

Verificando o estoque aqui... Sim, temos disponível no tamanho 36! É o modelo clássico de couro sintético, todo branco impecável, por R$ 299,90. Posso reservar um par para você agora? Ou quer mais detalhes, como opções de entrega? 😊
